# 03 - Fine-tune detection models

PRD section 3.8, flow 4. This version trains the **YOLO baseline** (Ultralytics) for mode 1 (real images only) and evaluates it. RT-DETR and the third model, and modes 2-5, are added to this notebook later.

Data layout (one folder per source, in Drive, or in a Kaggle dataset):

```
real/       images/*.jpg  labels/*.txt     # YOLO labels: class x_center y_center width height, all 0-1
synthetic/  images/*.jpg  labels/*.txt     # same layout
```

Class numbers follow the row order of `Data/objects_table.csv` (row 1 = class 0). Each source is split 80/20 (seed in `config.yaml`), then 10% of the 80% becomes the validation split that stops training early. **The headline number is `real_test`.** `synthetic_test` is reported too but is optimistic: synthetic images share backgrounds and style with the synthetic training images.

Runs on Colab (T4) or Kaggle (GPU on, Internet on). On Kaggle, upload `real/` and `synthetic/` as a dataset and set `KAGGLE_DATASET` below.

In [ ]:
BRANCH = "dev"  # set to your branch name to run a branch
TRAIN_ON = ["real"]  # mode 1. Add "synthetic" for modes 2-4 (the synthetic folder holds that mode's images)
KAGGLE_DATASET = "shelter-data"  # Kaggle only: name of the dataset that holds real/ and synthetic/

import os, subprocess, yaml
IN_COLAB, IN_KAGGLE = "COLAB_RELEASE_TAG" in os.environ, "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if os.path.exists("../config.yaml"):  # already inside a clone of the repo
    REPO = os.path.abspath("..")
else:
    REPO = ("/kaggle/working" if IN_KAGGLE else "/content") + "/genai-shelter-validator"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/AlmogSchwartzberg/genai-shelter-validator.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
cfg = yaml.safe_load(open(f"{REPO}/config.yaml"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DATA = cfg["drive_root"]
elif IN_KAGGLE:  # outputs go to /kaggle/working; download them (or "Save Version") before the session ends
    ROOT, DATA = "/kaggle/working/genai-shelter-validator", f"/kaggle/input/{KAGGLE_DATASET}"
else:
    ROOT = DATA = cfg["drive_root"]
OUT = f"{ROOT}/results"
os.makedirs(OUT, exist_ok=True)
print("Config loaded. Data:", DATA, "| Outputs:", OUT)

In [ ]:
%pip install -q ultralytics
import csv, glob, hashlib, random
import numpy as np, pandas as pd, torch
from PIL import Image, ImageDraw
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator. Training on CPU will be very slow.")
rows = list(csv.DictReader(open(f"{REPO}/Data/objects_table.csv")))
CLASSES = [r["name"] for r in rows]
ROLE = {r["name"]: r["role"] for r in rows}
LOOKALIKE_OF = {r["name"]: r["lookalike_of"] for r in rows if r["lookalike_of"]}
NC = len(CLASSES)
MODEL = next((m for m in cfg["detection"]["models"] if m.endswith(".pt")), None)
assert MODEL, "No YOLO weights file (*.pt) in detection.models in config.yaml"
IMGSZ = cfg["generation"]["image_size"]
print(f"{NC} classes:", CLASSES, "| model:", MODEL)

## 1. Load each source, check labels, split 80/20 (and 10% of the 80% for validation)

In [ ]:
def label_path(img):
    return img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

def read_labels(img):  # -> list of (class, xc, yc, w, h), raises on a bad line
    out = []
    for n, line in enumerate(open(label_path(img)).read().splitlines(), 1):
        if not line.strip():
            continue
        f = line.split()
        ok = len(f) == 5 and f[0].isdigit() and int(f[0]) < NC and all(0 <= float(v) <= 1 for v in f[1:])
        if not ok:
            raise ValueError(f"Bad label in {label_path(img)} line {n}: '{line}' (expected: class 0-{NC - 1}, then 4 numbers in 0-1)")
        out.append((int(f[0]), *map(float, f[1:])))
    return out

def split(source):
    folder = f"{DATA}/{source}"
    if not os.path.isdir(f"{folder}/images"):
        print(f"[{source}] not found at {folder}/images - skipped. Folders in {DATA}:", os.listdir(DATA) if os.path.isdir(DATA) else "missing")
        return None
    imgs = sorted(p for p in glob.glob(f"{folder}/images/*") if p.lower().endswith((".jpg", ".jpeg", ".png")))
    labeled = [p for p in imgs if os.path.exists(label_path(p))]
    if len(labeled) < len(imgs):
        print(f"[{source}] {len(imgs) - len(labeled)} images have no label file and are left out (PRD: missing label = not used)")
    for p in labeled:
        read_labels(p)  # validates every line
    random.Random(cfg["seed"]).shuffle(labeled)
    n_test = round(len(labeled) * cfg["split"]["test"])
    test, rest = labeled[:n_test], labeled[n_test:]
    n_val = round(len(rest) * cfg["split"]["val_from_train"])
    s = {"test": test, "val": rest[:n_val], "train": rest[n_val:]}
    md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
    if {md5(p) for p in test} & {md5(p) for p in rest}:
        print(f"[{source}] WARNING: the same image appears in both test and train (duplicate files). Remove duplicates before trusting results.")
    print(f"[{source}] {len(labeled)} labeled images -> train {len(s['train'])} / val {len(s['val'])} / test {len(s['test'])}")
    return s

SPLITS = {name: s for name in ("real", "synthetic") if (s := split(name))}
assert all(t in SPLITS for t in TRAIN_ON), f"TRAIN_ON {TRAIN_ON} needs these folders: {[t for t in TRAIN_ON if t not in SPLITS]}"
if "real" not in SPLITS:
    print("\nWARNING: no real images yet. Results below are synthetic-only and optimistic - do not report them as the PRD result.")

# Ultralytics reads a text file of image paths; labels are found by swapping /images/ for /labels/
os.makedirs(f"{OUT}/splits", exist_ok=True)
def write_list(name, paths):
    f = f"{OUT}/splits/{name}.txt"
    open(f, "w").write("\n".join(paths))
    return f

train_imgs = [p for s in TRAIN_ON for p in SPLITS[s]["train"]]
val_imgs = [p for s in TRAIN_ON for p in SPLITS[s]["val"]]
assert train_imgs and val_imgs, "Empty train or validation split - too few labeled images"
def data_yaml(name, train, val):
    f = f"{OUT}/splits/{name}.yaml"
    yaml.safe_dump({"train": write_list(f"{name}_train", train), "val": write_list(f"{name}_val", val),
                    "names": dict(enumerate(CLASSES))}, open(f, "w"))
    return f
TAG = f"{os.path.splitext(os.path.basename(MODEL))[0]}_{'+'.join(TRAIN_ON)}"
train_yaml = data_yaml(f"{TAG}_fit", train_imgs, val_imgs)
test_yamls = {f"{s}_test": data_yaml(f"{TAG}_{s}_test", train_imgs, SPLITS[s]["test"]) for s in SPLITS if SPLITS[s]["test"]}

counts = pd.DataFrame({f"{s}_{part}": pd.Series([c for p in imgs for c, *_ in read_labels(p)]).map(dict(enumerate(CLASSES))).value_counts()
                       for s in SPLITS for part, imgs in SPLITS[s].items()}).reindex(CLASSES).fillna(0).astype(int)
counts

## 2. Train (resumes after a disconnect; stops early on the validation split)

In [ ]:
RUN = f"yolo_{TAG}"
run_dir, last, best = f"{OUT}/{RUN}", f"{OUT}/{RUN}/weights/last.pt", f"{OUT}/{RUN}/weights/best.pt"
args = dict(
    data=train_yaml, epochs=cfg["detection"]["epochs"], patience=cfg["detection"]["patience"],
    imgsz=IMGSZ, batch=cfg["detection"]["batch_size"], device=DEVICE, workers=2,
    seed=cfg["seed"], deterministic=True, project=OUT, name=RUN, exist_ok=True,
    # PRD 3.8: Ultralytics' built-in augmentation is off, so mode 1 has no augmentation
    mosaic=0, close_mosaic=0, mixup=0, copy_paste=0, hsv_h=0, hsv_s=0, hsv_v=0,
    degrees=0, translate=0, scale=0, shear=0, perspective=0, flipud=0, fliplr=0, erasing=0,
    augmentations=[],  # also turns off Albumentations' default blur/grayscale/CLAHE
)
if os.path.exists(f"{run_dir}/done"):
    print("Already trained:", best)
else:
    try:
        if os.path.exists(last):
            print("Resuming from", last)
            YOLO(last).train(resume=True)
        else:
            YOLO(MODEL).train(**args)
    except torch.cuda.OutOfMemoryError:
        raise SystemExit("GPU out of memory. Lower detection.batch_size in config.yaml.")
    open(f"{run_dir}/done", "w").write("ok")
pd.read_csv(f"{run_dir}/results.csv").rename(columns=str.strip)[["epoch", "train/box_loss", "val/box_loss", "metrics/mAP50(B)"]].tail(5)

## 3. Evaluate: mAP@50 overall and per class, on each test split

In [ ]:
model = YOLO(best)
summary, per_class = [], []
for name, yml in test_yamls.items():
    source = name.split("_")[0]
    m = model.val(data=yml, imgsz=IMGSZ, batch=cfg["detection"]["batch_size"], device=DEVICE, plots=False,
                  project=OUT, name=f"{RUN}_eval_{name}", exist_ok=True, verbose=False)
    b = m.box
    summary.append({"test_set": name, "images": len(SPLITS[source]["test"]), "mAP50": b.map50, "mAP50-95": b.map, "precision": b.mp, "recall": b.mr})
    ap = {int(c): i for i, c in enumerate(b.ap_class_index)}
    for c, cname in enumerate(CLASSES):  # classes with no ground-truth in this test set get NaN
        i = ap.get(c)
        per_class.append({"test_set": name, "class": cname, "role": ROLE[cname],
                          "instances": counts.loc[cname, name],
                          "AP50": b.ap50[i] if i is not None else np.nan, "precision": b.p[i] if i is not None else np.nan,
                          "recall": b.r[i] if i is not None else np.nan, "F1": b.f1[i] if i is not None else np.nan})
summary, per_class = pd.DataFrame(summary), pd.DataFrame(per_class)
summary.to_csv(f"{OUT}/{RUN}_summary.csv", index=False)
per_class.to_csv(f"{OUT}/{RUN}_per_class.csv", index=False)
print("real_test is the number to report. synthetic_test is optimistic (shared backgrounds and style with training images).")
display(summary.round(3))
display(per_class.round(3))

## 4. Error analysis: where does the model fail?

In [ ]:
def iou(a, b):
    w, h = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1])
    inter = max(w, 0) * max(h, 0)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def analyse(imgs, conf=0.25):
    """Greedy match of predictions to ground truth at IoU 0.5. Confusion rows = true class (last row = background),
    columns = predicted class (last column = missed)."""
    cm, per_image = np.zeros((NC + 1, NC + 1), int), []
    for r in model.predict(imgs, conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False, stream=True):
        H, W = r.orig_shape
        gt = [(c, [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]) for c, xc, yc, w, h in read_labels(r.path)]
        preds = sorted(zip(r.boxes.conf.tolist(), r.boxes.cls.int().tolist(), r.boxes.xyxy.tolist()), reverse=True)
        free, errors = list(range(len(gt))), 0
        for _, pc, pbox in preds:
            best_i = max(free, key=lambda i: iou(gt[i][1], pbox), default=None)
            if best_i is not None and iou(gt[best_i][1], pbox) >= 0.5:
                free.remove(best_i)
                cm[gt[best_i][0], pc] += 1
                errors += gt[best_i][0] != pc
            else:
                cm[NC, pc] += 1  # prediction with no object there
                errors += 1
        for i in free:
            cm[gt[i][0], NC] += 1  # object the model missed
        errors += len(free)
        per_image.append((errors, r.path, gt, preds))
    return cm, sorted(per_image, key=lambda x: -x[0])

def show_failures(per_image, name, k=8):
    tiles = []
    for errors, path, gt, preds in per_image[:k]:
        im = Image.open(path).convert("RGB")
        d = ImageDraw.Draw(im)
        for c, box in gt:
            d.rectangle(box, outline="lime", width=2)
            d.text((box[0] + 2, box[1] + 2), CLASSES[c], fill="lime")
        for cf, c, box in preds:
            d.rectangle(box, outline="red", width=2)
            d.text((box[0] + 2, box[3] - 12), f"{CLASSES[c]} {cf:.2f}", fill="red")
        tiles.append(np.array(im.resize((IMGSZ, IMGSZ))))
    tiles += [np.zeros_like(tiles[0])] * (-len(tiles) % 4)
    grid = np.vstack([np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)])
    Image.fromarray(grid).save(f"{OUT}/{RUN}_{name}_failures.png")
    display(Image.fromarray(grid).resize((grid.shape[1] // 2, grid.shape[0] // 2)))

import matplotlib.pyplot as plt
for name in test_yamls:
    source = name.split("_")[0]
    cm, per_image = analyse(SPLITS[source]["test"])
    labels = CLASSES + ["background/missed"]
    pd.DataFrame(cm, index=["true " + l for l in labels[:-1]] + ["no object"], columns=["pred " + l for l in labels[:-1]] + ["missed"]).to_csv(f"{OUT}/{RUN}_{name}_confusion.csv")
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        if v:
            ax.text(j, i, v, ha="center", va="center", fontsize=8)
    ax.set_xticks(range(NC + 1), CLASSES + ["missed"], rotation=60, ha="right")
    ax.set_yticks(range(NC + 1), CLASSES + ["no object"])
    ax.set_xlabel("predicted (confidence >= 0.25)"); ax.set_ylabel("true")
    ax.set_title(f"{name}: confusion matrix"); fig.tight_layout()
    fig.savefig(f"{OUT}/{RUN}_{name}_confusion.png", dpi=150); plt.show()

    # The two project-level errors (PRD 3.9): lookalikes taken for the real thing, forbidden objects not detected
    idx = {c: i for i, c in enumerate(CLASSES)}
    rates = [{"error": f"{l} detected as {r}", "rate": cm[idx[l], idx[r]] / cm[idx[l], :].sum() if cm[idx[l], :].sum() else np.nan, "of_objects": cm[idx[l], :].sum()}
             for l, r in LOOKALIKE_OF.items()]
    rates += [{"error": f"{c} not detected as {c}", "rate": 1 - cm[idx[c], idx[c]] / cm[idx[c], :].sum() if cm[idx[c], :].sum() else np.nan, "of_objects": cm[idx[c], :].sum()}
              for c in CLASSES if ROLE[c] == "forbidden"]
    rates = pd.DataFrame(rates)
    rates.to_csv(f"{OUT}/{RUN}_{name}_error_rates.csv", index=False)
    print(f"{name}: false confirmations of lookalikes and missed forbidden objects (a small 'of_objects' means the rate is noisy)")
    display(rates.round(3))
    show_failures(per_image, name)